# Final test: LRW-pretrained frontend + balanced BiGRU

Модель и протокол уже выбраны по validation до запуска этого notebook.

```text
dataset v2
-> frozen LRW-pretrained 3D Conv + ResNet18
-> BiGRU checkpoints for seeds 42, 123, 2026
-> one final evaluation on reserved test speakers
```

Здесь нет обучения, optimizer, early stopping или подбора гиперпараметров.
После получения test metrics модель нельзя менять на основании test.

## 1. Подключить Drive и получить upstream code

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%pip -q install gdown

In [ ]:
from pathlib import Path
import subprocess
import sys

import gdown

drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
experiment_dir = drive_root / 'controlled_lrw_frontend_bigru_v2'
final_dir = drive_root / 'final_lrw_frontend_test_v2'
final_dir.mkdir(parents=True, exist_ok=True)

upstream_dir = Path('/content/Lipreading_using_Temporal_Convolutional_Networks')
if not upstream_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git',
        str(upstream_dir),
    ], check=True)
sys.path.insert(0, str(upstream_dir))

checkpoint_path = (
    drive_root / 'upstream_lrw_pretrained' / 'lrw_resnet18_mstcn_video.pth'
)
if not checkpoint_path.exists():
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    downloaded = gdown.download(
        id='1vqMpxZ5LzJjg50HlZdj_QFJGm2gQmDUD',
        output=str(checkpoint_path),
        quiet=False,
    )
    if downloaded is None:
        raise RuntimeError('Official LRW checkpoint download failed.')

seeds = [42, 123, 2026]
model_paths = {
    seed: experiment_dir / f'lrw_frontend_bigru_v2_seed{seed}_best.pt'
    for seed in seeds
}
missing_models = [str(path) for path in model_paths.values() if not path.exists()]
assert not missing_models, f'Missing selected validation checkpoints: {missing_models}'

print('official checkpoint:', checkpoint_path)
print('selected model checkpoints:', model_paths)
print('NO TRAINING WILL BE RUN.')

## 2. Восстановить dataset v2 и открыть только test metadata

In [ ]:
import shutil
import zipfile

extract_dir = Path('/content/lipreading_data_v2')
relative_split_dir = (
    Path('03_splits')
    / 'ml_splits_mouth_crops_padded_dataset_v2'
    / 'speaker_top10'
)
split_dir = extract_dir / 'ru_dataset' / relative_split_dir

if not split_dir.exists():
    matches = list(Path('/content/drive').rglob('colab_lipreading_dataset_v2.zip'))
    if not matches:
        matches = list(Path('/content/drive').rglob('kaggle_lipreading_dataset_v2.zip'))
    if not matches:
        raise FileNotFoundError('Dataset v2 archive was not found on Google Drive.')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0], 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print('Dataset restored from:', matches[0])

data_root = extract_dir / 'ru_dataset'
print('split:', split_dir)

In [ ]:
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
test_df = pd.read_csv(split_dir / 'test.csv')
vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = {index: word for word, index in word_to_idx.items()}

assert len(test_df) == 150
assert sorted(test_df['speaker_id'].unique()) == ['spk07', 'spk08']
assert all((data_root / path).is_file() for path in test_df['clip_path'])

print('device:', device)
print('FINAL TEST clips:', len(test_df))
print('FINAL TEST speakers:', sorted(test_df['speaker_id'].unique()))
print('vocab:', vocab)

## 3. Извлечь LRW-pretrained features для test один раз

In [ ]:
def load_lrw_video(path, num_frames=24, input_size=96, crop_size=88):
    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frames = []
    for frame_index in np.linspace(0, total_frames - 1, num_frames).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (input_size, input_size))
        offset = (input_size - crop_size) // 2
        frame = frame[offset:offset + crop_size, offset:offset + crop_size]
        frame = frame.astype(np.float32) / 255.0
        frames.append((frame - 0.421) / 0.165)
    cap.release()

    if not frames:
        raise ValueError(f'No frames loaded: {path}')
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())
    return torch.from_numpy(np.stack(frames)).float().unsqueeze(0)


class TestVideoDataset(Dataset):
    def __len__(self):
        return len(test_df)

    def __getitem__(self, index):
        row = test_df.iloc[index]
        return load_lrw_video(data_root / row['clip_path']), index


test_video_loader = DataLoader(
    TestVideoDataset(), batch_size=4, shuffle=False, num_workers=2
)

from lipreading.model import Lipreading

tcn_options = {
    'num_layers': 4,
    'kernel_size': [3, 5, 7],
    'dropout': 0.2,
    'dwpw': False,
    'width_mult': 1,
}
lrw_encoder = Lipreading(
    modality='video', num_classes=500, tcn_options=tcn_options,
    backbone_type='resnet', relu_type='swish', width_mult=1.0,
    use_boundary=False, extract_feats=True,
)
upstream_checkpoint = load_torch_file(checkpoint_path)
lrw_encoder.load_state_dict(
    upstream_checkpoint.get('model_state_dict', upstream_checkpoint), strict=True
)
for parameter in lrw_encoder.parameters():
    parameter.requires_grad = False
lrw_encoder = lrw_encoder.to(device).eval()

test_cache_path = final_dir / 'dataset_v2_test_lrw_resnet18_features.pt'
if test_cache_path.exists():
    test_cache = load_torch_file(test_cache_path)
    print('loaded test cache:', test_cache_path)
else:
    features = []
    start = time.perf_counter()
    with torch.no_grad():
        for batch_index, (videos, indices) in enumerate(test_video_loader, start=1):
            videos = videos.to(device)
            batch_features = lrw_encoder(
                videos, lengths=[videos.shape[2]] * videos.shape[0]
            )
            features.append(batch_features.cpu())
            if batch_index == 1 or batch_index % 10 == 0:
                print(f'test feature batches: {batch_index}/{len(test_video_loader)}')
    test_cache = {
        'features': torch.cat(features),
        'words': test_df['word'].tolist(),
        'speakers': test_df['speaker_id'].tolist(),
        'feature_extraction_seconds': time.perf_counter() - start,
        'encoder': 'official LRW-pretrained 3D Conv + ResNet18',
    }
    torch.save(test_cache, test_cache_path)
    print('saved test cache:', test_cache_path)

assert test_cache['features'].shape == (150, 24, 512)
assert test_cache['words'] == test_df['word'].tolist()
print('test features:', test_cache['features'].shape)

## 4. Восстановить три заранее выбранных BiGRU checkpoint

In [ ]:
class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


class TestFeatureDataset(Dataset):
    def __init__(self):
        self.features = test_cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in test_cache['words']], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


test_loader = DataLoader(
    TestFeatureDataset(), batch_size=32, shuffle=False, num_workers=0
)


def classification_metrics(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1

    class_rows = []
    for class_id in range(num_classes):
        tp = int(matrix[class_id, class_id])
        support = int(matrix[class_id].sum())
        predicted = int(matrix[:, class_id].sum())
        precision = tp / predicted if predicted else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        class_rows.append({
            'class_id': class_id,
            'word': idx_to_word[class_id],
            'support': support,
            'predicted_count': predicted,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })

    active = [row for row in class_rows if row['support']]
    return {
        'accuracy': float(np.trace(matrix) / matrix.sum()),
        'macro_f1': float(np.mean([row['f1'] for row in active])),
        'balanced_accuracy': float(np.mean([row['recall'] for row in active])),
        'predicted_classes': int(np.count_nonzero(matrix.sum(axis=0))),
        'class_rows': class_rows,
        'confusion_matrix': matrix,
    }


def evaluate_checkpoint(seed):
    saved = load_torch_file(model_paths[seed])
    assert saved['vocab'] == vocab
    model = BiGRUClassifier(len(vocab)).to(device)
    model.load_state_dict(saved['model_state_dict'], strict=True)
    model.eval()

    preds = []
    labels = []
    with torch.no_grad():
        for features, targets in test_loader:
            outputs = model(features.to(device))
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(targets.tolist())

    metrics = classification_metrics(preds, labels, len(vocab))
    row = {
        'seed': seed,
        'test_accuracy': metrics['accuracy'],
        'test_macro_f1': metrics['macro_f1'],
        'test_balanced_accuracy': metrics['balanced_accuracy'],
        'predicted_classes': metrics['predicted_classes'],
        'test_clips': len(labels),
        'test_speakers': 2,
    }
    prediction_table = pd.DataFrame({
        'sample_index': range(len(labels)),
        'speaker_id': test_cache['speakers'],
        'true_word': [idx_to_word[index] for index in labels],
        'predicted_word': [idx_to_word[index] for index in preds],
    })
    prediction_table.to_csv(final_dir / f'test_predictions_seed{seed}.csv', index=False)
    pd.DataFrame(metrics['class_rows']).to_csv(
        final_dir / f'test_per_class_seed{seed}.csv', index=False
    )
    return row, metrics


print('Evaluation code ready. There is no optimizer or training loop.')

## 5. Единственное финальное test evaluation

In [ ]:
rows = []
metrics_by_seed = {}
for seed in seeds:
    row, metrics = evaluate_checkpoint(seed)
    rows.append(row)
    metrics_by_seed[seed] = metrics
    print(
        f"seed={seed} | test acc={row['test_accuracy']:.4f} | "
        f"macro-F1={row['test_macro_f1']:.4f} | "
        f"balanced_acc={row['test_balanced_accuracy']:.4f} | "
        f"classes={row['predicted_classes']}/10"
    )

test_runs = pd.DataFrame(rows).sort_values('seed')
test_runs.to_csv(final_dir / 'final_test_multiseed_runs.csv', index=False)
display(test_runs)

## 6. Финальная таблица, per-class recall и confusion matrix

In [ ]:
metric_columns = [
    'test_accuracy',
    'test_macro_f1',
    'test_balanced_accuracy',
    'predicted_classes',
]
summary = test_runs[metric_columns].agg(['mean', 'std']).T
summary.columns = ['mean', 'std']
summary.reset_index(names='metric').to_csv(
    final_dir / 'final_test_multiseed_summary.csv', index=False
)
display(summary)

per_class_frames = []
combined_matrix = np.zeros((len(vocab), len(vocab)), dtype=int)
for seed, metrics in metrics_by_seed.items():
    frame = pd.DataFrame(metrics['class_rows'])
    frame['seed'] = seed
    per_class_frames.append(frame)
    combined_matrix += metrics['confusion_matrix']

per_class_all = pd.concat(per_class_frames, ignore_index=True)
per_class_summary = per_class_all.groupby('word')[['precision', 'recall', 'f1']].agg(['mean', 'std'])
per_class_summary.to_csv(final_dir / 'final_test_per_class_summary.csv')
display(per_class_summary)

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(combined_matrix, cmap='Blues')
ax.set_title('Final test confusion matrix, summed across three seeds')
ax.set_xlabel('predicted word')
ax.set_ylabel('true word')
ax.set_xticks(range(len(vocab)), vocab, rotation=45, ha='right')
ax.set_yticks(range(len(vocab)), vocab)
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(final_dir / 'final_test_confusion_matrix_three_seeds.png', dpi=150)
plt.show()

print('FINAL TEST COMPLETED.')
print('Freeze these results. Do not tune the model using test metrics.')
print('Results saved to:', final_dir)